In [63]:
import os   
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from google import genai
from google.genai import types
import gradio as gr
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

True

In [64]:
model = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    temperature = 1.0,
    max_tokens = 2000,
    timeout = None,
    max_retries = 2,
)

In [65]:
def document_loader(file):
    loader = PyPDFLoader(file)
    loaded_document = loader.load()
    return loaded_document

def text_splitter(data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 420,
        chunk_overlap = 50,
        length_function=len,
    )
    chunks = text_splitter.split_documents(data)
    return chunks

def vc_db(chunks):
    embedding_model = GoogleGenerativeAIEmbeddings(
      model="gemini-embedding-2-preview"
    )
    vectordb = Chroma.from_documents(chunks, embedding_model)
    return vectordb
def retriever(file):
    splits = document_loader(file)
    chunks = text_splitter(splits)
    vectordb = vc_db(chunks)
    retriever = vectordb.as_retriever()
    return retriever

In [66]:
def retriever_qa(file, query):
    llm = model
    retriever_obj = retriever(file)
    
    # 1. Define how you want to prompt the LLM
    template = """Answer the question based only on the following context:
    {context}
    
    Question: {question}
    """
    prompt = ChatPromptTemplate.from_template(template)
    
    # 2. Define a helper function to format the retrieved documents 
    # (This replaces the old chain_type="stuff" behavior)
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)
    
    # 3. Build the LCEL Chain
    qa_chain = (
        {"context": retriever_obj | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    
    # 4. Invoke the chain directly with the query string
    response = qa_chain.invoke(query)
    
    return response

In [ ]:
rag_application = gr.Interface(
    fn=retriever_qa,
    flagging_mode="never",
    inputs=[
        gr.File(
            label="Upload PDF File",
            file_count="single",
            file_types=[".pdf"],
            type="filepath", 
        ),
        gr.Textbox(
            label="Input Query",
            lines=2,
            placeholder="Type your question here...",
        ),
    ],
    outputs=gr.Markdown(label="Output"),
    title="RAG Chatbot",
    description="Upload a PDF document and ask any question.",
)


In [73]:
rag_application.close()

Closing server running on port: 7861


In [75]:
rag_application.launch(server_port=7861,debug=True,show_error=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----


ValueError: When localhost is not accessible, a shareable link must be created. Please set share=True or check your proxy settings to allow access to localhost.

In [ ]:
#Finally end :))